# 4.3 — منحنى ROC وتحليل AUC — D-MorphNet

هذا الدفتر ينفّذ قسم **"ROC Curve and AUC Analysis"** ويعيد إنتاج **الشكل ٣** بنفس أسلوب الورقة.

## ما الذي يقيسه منحنى ROC؟

المنحنى يرسم العلاقة بين معدلين عند **كل عتبة قرار ممكنة** لدالة $w^TF+b$:

$$TPR = \frac{TP}{TP + FN} \qquad \text{(معدل اكتشاف المورف الصحيح)}$$

$$FPR = \frac{FP}{FP + TN} \qquad \text{(معدل الإنذار الخاطئ على الوجوه الحقيقية)}$$

- كل نقطة على المنحنى = عتبة مختلفة: خفض العتبة يكشف مورفات أكثر (TPR أعلى) لكن بإنذارات خاطئة أكثر (FPR أعلى).
- **AUC (المساحة تحت المنحنى)**: ملخص بقيمة واحدة — في الورقة بلغت **0.965**، والقيمة القريبة من 1.0 تعني تمييزاً ممتازاً بين الوجوه الحقيقية والمدموجة.
- **الارتفاع الحاد قرب نقطة البداية** في الشكل ٣ يعني أن النظام يكتشف معظم صور المورف مع إبقاء الإنذارات الخاطئة منخفضة جداً — وهذه أهم خاصية عملياً في الأنظمة البيومترية.


## ١) تجهيز البيئة


In [ ]:
!pip -q install scikit-learn joblib pandas

import os, glob, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PAPER_AUC = 0.965        # قيمة AUC المنشورة في الورقة (الشكل 3)
print("✅ البيئة جاهزة")


## ٢) تحميل السمات والمصنّف وحساب درجات القرار

نحمّل سمات الاختبار ومصنّف SVM النهائي (من الدفاتر السابقة، محلياً أو من Drive)، ثم نحسب درجة القرار $w^TF+b$ لكل صورة اختبار — هذه الدرجات المتصلة هي أساس بناء منحنى ROC.


In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

FEAT_DIR = "/content/DMorphNet_features"
DRIVE_DIR = "/content/drive/MyDrive/DMorphNet_models"


def find_file(name):
    for root in [FEAT_DIR, DRIVE_DIR]:
        p = os.path.join(root, name)
        if os.path.exists(p):
            return p
    return None


if find_file("effb6_ft_train.npz") is None and find_file("effb6_train.npz") is None:
    from google.colab import drive
    drive.mount('/content/drive')

PREFIX = "effb6_ft" if find_file("effb6_ft_train.npz") else "effb6"
F, y = {}, {}
for split in ["train", "test"]:
    data = np.load(find_file(f"{PREFIX}_{split}.npz"))
    F[split] = data["X"].astype(np.float32)
    y[split] = np.where(data["y"] == 0, -1, +1)      # -1 حقيقية ، +1 مدموجة

svm_path, sc_path = find_file("svm_hybrid_final.joblib"), find_file("scaler_hybrid_final.joblib")
if svm_path and sc_path:
    svm_final, scaler = joblib.load(svm_path), joblib.load(sc_path)
    print("تم تحميل المصنّف النهائي المحفوظ")
else:
    print("لا يوجد مصنّف محفوظ — إعادة تدريب سريعة ...")
    scaler = StandardScaler().fit(F["train"])
    svm_final = LinearSVC(C=1.0, random_state=SEED)
    svm_final.fit(scaler.transform(F["train"]), y["train"])

# درجة القرار المتصلة w^T F + b لكل صورة اختبار
scores = svm_final.decision_function(scaler.transform(F["test"]))
print(f"عدد صور الاختبار: {len(scores)} — "
      f"مدى الدرجات: [{scores.min():.2f}, {scores.max():.2f}]")


## ٣) حساب نقاط منحنى ROC وقيمة AUC

`roc_curve` تجرّب كل العتبات الممكنة وتحسب (FPR, TPR) عند كل واحدة، ثم `auc` تحسب المساحة تحت المنحنى بطريقة شبه المنحرف.


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y["test"], scores, pos_label=+1)
roc_auc = auc(fpr, tpr)

print(f"عدد نقاط المنحنى (عتبات مختلفة): {len(fpr)}")
print(f"AUC لنظامنا  = {roc_auc:.4f}")
print(f"AUC في الورقة = {PAPER_AUC}")


## ٤) رسم الشكل ٣ بنفس أسلوب الورقة

نفس عناصر الشكل المنشور تماماً: منحنى أزرق متصل، قطر برتقالي متقطع (التخمين العشوائي)، العنوان "ROC Curve"، تسميات المحاور بالإنجليزية، ووسم `AUC=...` في الزاوية السفلية اليمنى، مع شبكة خفيفة — ونحفظه بدقة عالية للاستخدام في التقرير.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.plot(fpr, tpr, color="tab:blue", linewidth=2.2,
        label=f"AUC={roc_auc:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="tab:orange", linewidth=1.8)
ax.set_title("ROC Curve", fontsize=14)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
ax.grid(alpha=0.35)
ax.legend(loc="lower right", fontsize=11, frameon=True)
plt.tight_layout()

FIG_PATH = "/content/fig3_roc_curve.png"
plt.savefig(FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()
print("تم حفظ الشكل عالي الدقة في:", FIG_PATH)


## ٥) تحليل "الارتفاع الحاد قرب البداية"

نقيس عملياً ما تصفه الورقة: كم نسبة المورفات المكتشفة (TPR) عندما نقيّد الإنذارات الخاطئة (FPR) بقيم منخفضة جداً؟ جدول العتبات يوضح المقايضة عند نقاط تشغيل مختلفة — مفيد لاختيار عتبة النشر الفعلية حسب حساسية التطبيق.


In [ ]:
def tpr_at_fpr(target_fpr):
    # أعلى TPR ممكن دون تجاوز FPR المستهدف
    mask = fpr <= target_fpr
    return tpr[mask].max() if mask.any() else 0.0


print("قدرة الكشف عند معدلات إنذار خاطئ منخفضة:")
rows = []
for f_target in [0.01, 0.05, 0.10, 0.20]:
    t = tpr_at_fpr(f_target)
    rows.append({"FPR المسموح": f"{f_target:.0%}",
                 "TPR المحقق (نسبة المورف المكتشف)": f"{t:.1%}"})
    print(f"  عند FPR ≤ {f_target:.0%} → يكتشف النظام {t:.1%} من صور المورف")

pd.DataFrame(rows)


## ٦) نقطة تساوي الخطأ (EER) والعتبة المثلى

مقياسان إضافيان شائعان في الأنظمة البيومترية يُشتقان من نفس المنحنى:
- **EER (Equal Error Rate)**: النقطة التي يتساوى عندها معدل الإنذار الخاطئ مع معدل الفشل في الكشف $(FPR = 1 - TPR)$ — كلما صغرت كان النظام أفضل.
- **العتبة المثلى بمعيار Youden**: العتبة التي تعظّم $J = TPR - FPR$ — أفضل توازن بين الكشف والإنذارات الخاطئة.


In [ ]:
# نقطة تساوي الخطأ EER
fnr = 1 - tpr
eer_idx = int(np.argmin(np.abs(fpr - fnr)))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2

# العتبة المثلى بمعيار Youden J = TPR - FPR
j_idx = int(np.argmax(tpr - fpr))

print(f"نقطة تساوي الخطأ EER = {eer:.4f} (عند العتبة {thresholds[eer_idx]:.3f})")
print(f"العتبة المثلى (Youden) = {thresholds[j_idx]:.3f} → "
      f"TPR = {tpr[j_idx]:.3f} ، FPR = {fpr[j_idx]:.3f}")

# رسم المنحنى مع النقطتين
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.plot(fpr, tpr, color="tab:blue", linewidth=2.2, label=f"AUC={roc_auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="tab:orange", linewidth=1.8)
ax.scatter(fpr[j_idx], tpr[j_idx], s=90, color="green", zorder=5,
           label=f"العتبة المثلى (J) — TPR={tpr[j_idx]:.2f}")
ax.scatter(fpr[eer_idx], tpr[eer_idx], s=90, color="red", zorder=5,
           label=f"نقطة EER = {eer:.3f}")
ax.set_title("ROC Curve — نقاط التشغيل المهمة")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.grid(alpha=0.35)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## ٧) حفظ الشكل والنتائج في Google Drive

نحفظ الشكل ٣ عالي الدقة (PNG بدقة 300) ونقاط المنحنى كاملة (CSV) وملخص التحليل — جاهزة للإدراج في التقرير أو العرض.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = "/content/drive/MyDrive/DMorphNet_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

shutil.copy(FIG_PATH, os.path.join(RESULTS_DIR, "fig3_roc_curve.png"))

pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": thresholds}).to_csv(
    os.path.join(RESULTS_DIR, "roc_points.csv"), index=False)

pd.DataFrame([{
    "AUC (ours)": round(roc_auc, 4),
    "AUC (paper)": PAPER_AUC,
    "EER": round(eer, 4),
    "Best threshold (Youden)": round(float(thresholds[j_idx]), 4),
    "TPR@FPR=5%": round(tpr_at_fpr(0.05), 4),
    "TPR@FPR=10%": round(tpr_at_fpr(0.10), 4),
}]).to_csv(os.path.join(RESULTS_DIR, "roc_auc_summary.csv"), index=False)

print("تم الحفظ في:", RESULTS_DIR)
print(os.listdir(RESULTS_DIR))
print("\n🎉 اكتمل قسم 4.3 — منحنى ROC وتحليل AUC")
